In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


## Local MCP server

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "stdio",
                "command": "python",
                "args": ["resources/2.1_mcp_server.py"],
            }
    }
)

In [4]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [5]:
from langchain.agents import create_agent

from langchain_ollama import ChatOllama
model = ChatOllama(model="gemma4:e2b")

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=prompt
)

In [6]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

In [7]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Tell me about the langchain-mcp-adapters library', additional_kwargs={}, response_metadata={}, id='0fb97d38-b800-4e17-a8fc-dd16a2442175'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e2b', 'created_at': '2026-04-22T14:18:23.9227784Z', 'done': True, 'done_reason': 'stop', 'total_duration': 62737942300, 'load_duration': 24188924800, 'prompt_eval_count': 210, 'prompt_eval_duration': 9137635300, 'eval_count': 253, 'eval_duration': 28666724400, 'logprobs': None, 'model_name': 'gemma4:e2b', 'model_provider': 'ollama'}, id='lc_run--019db58d-79f8-7ec1-a488-0aab39645e77-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'langchain-mcp-adapters library'}, 'id': 'b42f9d9e-8184-49f8-aa06-807bd6fc4c98', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 210, 'output_tokens': 253, 'total_tokens': 463}),
              ToolMessage(content=[{'type': 'text', 'text': '{\n  "query": "lang

## Online MCP

In [10]:
client = MultiServerMCPClient(
    {
        "travel_server": {
            "transport": "streamable_http",
            "url": "https://mcp.kiwi.com",
        }
    }
)

tools = await client.get_tools()

In [12]:
agent = create_agent(
    model=model,
    tools=tools,
    system_prompt="You are a travel agent. No follow questions."
)

In [16]:
question = HumanMessage(content="Get me a direct flight from San Francisco to Tokyo on September 30st")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)

McpError: MCP error -32602: MCP error -32602: Invalid arguments for tool search-flight: [
  {
    "code": "custom",
    "message": "Dates must be in the future. Current date is 22/04/2026",
    "path": [
      "departureDate"
    ]
  }
]